# Zebrafish 3D-4i embryos, opened cold

This notebook opens the store using nothing but the store itself: no ingest code and no access to the source archive. Everything below comes from the store's own metadata and pixels.

The repo is `earthmover-biology/zebrafish-3d-4i`. You need an Arraylake login with read access (`al auth login`). The store is a copy of the study's plate (BioImage Archive S-BIAD3228), written as an OME-Zarr 0.6 HCS plate: one well per cell of the source plate, one image per embryo.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import zarr
from arraylake import Client

repo = Client().get_repo("earthmover-biology/zebrafish-3d-4i")
root = zarr.open_group(repo.readonly_session("main").store, mode="r")
plate = root.attrs["ome"]["plate"]
source = root.attrs["source"]

print(plate["name"])
print(f"{len(plate['wells'])} wells, rows {[r['name'] for r in plate['rows']]}, columns {[c['name'] for c in plate['columns']]}")
print()
for key in ["study", "preprint", "license", "layout"]:
    print(f"{key}: {source[key]}")

## One well

A well holds one image per embryo. Its `scene` places each image on the microscope stage, in micrometers, in a coordinate system called `well`. The translations are `(c, z, y, x)`.

In [ ]:
well = root["C/02"]
images = [i["path"] for i in well.attrs["ome"]["well"]["images"]]
scene = well.attrs["ome"]["scene"]
placement = {t["input"]["path"]: t["translation"] for t in scene["coordinateTransformations"]}

print(f"well C/02 holds images {images}")
for path, (_, _, y, x) in placement.items():
    print(f"  image {path}: y = {y:7.1f} µm, x = {x:7.1f} µm")

## One embryo

An image is a z-stack with 16 channels, `(c, z, y, x)`. Each level is a coarser copy, and its `scale` says how big a pixel is.

In [ ]:
image = well["0"]
ms = image.attrs["ome"]["multiscales"][0]
axes = [a["name"] for a in ms["coordinateSystems"][0]["axes"]]

print(ms["name"], "axes:", axes)
for ds in ms["datasets"]:
    scale = ds["coordinateTransformations"][0]["transformations"][0]["scale"]
    array = image[ds["path"]]
    print(f"level {ds['path']}: shape {array.shape} {array.dtype}, pixel {scale[3]} µm in x and y, {scale[1]} µm in z, chunk {array.chunks}")

Each embryo was stained and imaged in four rounds of four channels. Channel `4 * r + c` is channel `c` of round `r`. The image's annotations record the round of every channel.

In [ ]:
channels = image.attrs["annotations"]["channels"]
names = [c["label"] for c in channels]
for r in range(4):
    print(f"round {r}:", [c["label"] for c in channels if c["round"] == r])

print("labels:", sorted(image["labels"].keys()))

Round 3 stains different markers in different wells. The root's `source.rounds` lists round 3's channels for each well.

In [ ]:
round3 = source["rounds"]["3"]
for stain in sorted({tuple(v) for v in round3.values()}):
    wells = [w for w, v in round3.items() if tuple(v) == stain]
    print(f"{len(wells):2d} wells: {list(stain)}")

## One plane

The middle z plane of level 2 (2.6 µm pixels) in three channels, and the `cells` label drawn over DAPI. Each panel reads one chunk.

In [ ]:
level = image["2"]
z = level.shape[1] // 2
px = ms["datasets"][2]["coordinateTransformations"][0]["transformations"][0]["scale"][3]
extent = [0, level.shape[3] * px, level.shape[2] * px, 0]

shown = ["DAPI_0", "Pol-II-S2P", "bCatenin"]
planes = {name: level[names.index(name), z] for name in shown}
cells = image["labels/cells/2"][z]

fig, axs = plt.subplots(1, 4, figsize=(16, 4.4), layout="constrained")
for ax, (name, plane) in zip(axs, planes.items()):
    ax.imshow(plane, cmap="gray", extent=extent, vmax=np.percentile(plane, 99.8))
    ax.set(title=name, xlabel="x (µm)", ylabel="y (µm)")
axs[3].imshow(planes["DAPI_0"], cmap="gray", extent=extent, vmax=np.percentile(planes["DAPI_0"], 99.8))
axs[3].imshow(np.ma.masked_equal(cells, 0) % 20, cmap="tab20", alpha=0.5, extent=extent, interpolation="nearest")
axs[3].set(title=f"DAPI_0 with cells ({len(np.unique(cells)) - 1} labels)", xlabel="x (µm)")
fig.suptitle(f"{ms['name']}, z = {z} µm");

## The well's embryos on the stage

The coarsest level (5.2 µm pixels) of every image in the well, DAPI_0 at the middle plane, placed at the translations from the well's `scene`. This reads one chunk per embryo.

In [ ]:
box = well.attrs["annotations"]["well_ROI_table"]["rows"]["Well"]
fig, ax = plt.subplots(figsize=(8, 7.5), layout="constrained")
for path, (_, _, y0, x0) in placement.items():
    coarse = well[path]["3"]
    plane = coarse[0, coarse.shape[1] // 2]
    size = image["0"].shape[2] * 0.65
    ax.imshow(plane, cmap="gray", vmax=np.percentile(plane, 99.8), extent=[x0, x0 + size, y0 + size, y0])
    ax.text(x0 + 20, y0 + 20, path, color="yellow", va="top")
ax.set(xlim=(0, box["len_x_micrometer"]), ylim=(box["len_y_micrometer"], 0), facecolor="black", xlabel="x (µm)", ylabel="y (µm)",
       title="Well C/02: DAPI_0 of each embryo at its stage position");